# Data Preprocessing of Cold Storage Room Environmental Data

## Import of Required Packages

In [1]:
import os
import csv
import pandas as pd

## Environmental Variables

In [2]:
PATH_RAW = '../Data/Raw'
PATH_PREPROCESSED = '../Data/Preprocessed'
FILE_PROTOCOL = '../Data/Versuchsablauf.xlsx'
FILE_OVERVIEW = '../Data/overview.csv'

## Filtering Datasets

In [3]:
# detection of raw datasets
datasets_raw = [file[0:8] for file in os.listdir(PATH_RAW) if os.path.isfile(os.path.join(PATH_RAW, file))]
# detection of preprocessed datasets
datasets_preprocessed = [file[0:8] for file in os.listdir(PATH_PREPROCESSED) if os.path.isfile(os.path.join(PATH_PREPROCESSED, file))]

# remove raw datasets in list if already preprocessed
datasets_raw = [dataset for dataset in datasets_raw if dataset not in datasets_preprocessed]

sensors_movable = ["SENSOR01", "SENSOR02"]
sensors_permanent = sorted(list(set(datasets_preprocessed).difference(set(sensors_movable))))

## Removing Multiplicate Headlines

In [5]:
for dataset in datasets_raw:
    # set paths to rawdata file and new preprocessed data file 
    current_raw = PATH_RAW + "/" + dataset + ".csv"
    current_preprocessed = PATH_PREPROCESSED + "/" + dataset + "_preprocessed.csv"
    with open(current_raw) as file_raw, open(current_preprocessed, 'w', newline='') as file_preprocessed:
        # read rawdata file
        csv_reader = csv.reader(file_raw, delimiter=';')
        # write preprocessed data file
        csv_writer = csv.writer(file_preprocessed, delimiter=';')
        # write header once
        csv_writer.writerow(['Date', 'Time', 'Temperature', 'Humidity'])
        # write only rows without header in preprocessed data file
        for row in csv_reader:
            if 'Date' not in row:
                csv_writer.writerow(row)

## Determining Start and Endpoints of Datasets

In [9]:
# detection of preprocessed datasets
datasets_preprocessed = [file[0:8] for file in os.listdir(PATH_PREPROCESSED) if os.path.isfile(os.path.join(PATH_PREPROCESSED, file))]
# initialize an empty data frame
datasets_overview = pd.DataFrame(data=None, columns=['Sensor', 'Start', 'End'])

for dataset in datasets_preprocessed:
    # construct the full path to the preprocessed CSV file for the current dataset
    path_file = os.path.join(PATH_PREPROCESSED, f"{dataset}_preprocessed.csv")
    
    # read the CSV file into data frame, using ';' as the separator and the first row as header
    data = pd.read_csv(path_file, sep=";", header=0)
    
    # add a new row to the data frame with the dataset identifier, the start timestamp and the end timestamp
    datasets_overview.loc[len(datasets_overview.index)] = [dataset, pd.to_datetime(data.iloc[0,0] + data.iloc[0,1], format='%d.%m.%Y%H:%M:%S'), pd.to_datetime(data.iloc[-1,0] + data.iloc[-1,1], format='%d.%m.%Y%H:%M:%S')]
    
datasets_overview = datasets_overview.transpose()
datasets_overview.columns = datasets_overview.iloc[0]  # Set the first row as columns
datasets_overview = datasets_overview.drop(0)  # Drop the first row and reset the index

datasets_overview.to_csv('overview.csv', sep=";")

datasets_filtered_complete = datasets_overview[(datasets_overview['Start'] < (datasets_overview['Start'].median() + pd.to_timedelta(10, "m"))) & (datasets_overview['End'] > (datasets_overview['End'].median() - pd.to_timedelta(10, "m")))]

# determine the latest start date among all datasets
start_max = max(datasets_filtered_complete['Start'])
# print(start_max)

# determine the earliest end date among all datasets
end_min = min(datasets_filtered_complete['End'])
# print(end_min)

2024-09-02 09:35:25
2024-09-11 11:24:55


## Cutting Datasets to Align Periods

In [10]:
datasets_complete = datasets_filtered_complete['Sensor'].to_list()
datasets_length = dict()

for dataset in datasets_complete:
    # construct the full path to the preprocessed CSV file for the current dataset
    path_file = os.path.join(PATH_PREPROCESSED, f"{dataset}_preprocessed.csv")
    
    # read the CSV file into data frame, using ';' as the separator and the first row as header
    data = pd.read_csv(path_file, sep=";", header=0)
    
    # filter rows to keep only those within the date range [start_max, end_min]
    data = data[(pd.to_datetime(data['Date'] + data['Time'], format='%d.%m.%Y%H:%M:%S') >= start_max) & (pd.to_datetime(data['Date'] + data['Time'], format='%d.%m.%Y%H:%M:%S') <= end_min)]
    
    datasets_length[dataset] = len(data)

    # save the filtered data back to the preprocessed CSV file
    data.to_csv(path_file, sep=";", index=False)
    
for key, value in datasets_length.items():
    if value < max(datasets_length.values()):
        datasets_complete.remove(key)

## Adding NaNs for the missing values of the partly failed sensors

In [11]:
#Add NaN values to the datasets that do not have data for the entire period
datasets_incomplete = sorted(list(set(datasets_preprocessed).difference(set(datasets_complete))))

datasets_missing = pd.DataFrame(data=None, columns=['Sensor', 'Missing Start', 'Missing End', 'Length'])

for dataset in datasets_incomplete:
    
    path_file = os.path.join(PATH_PREPROCESSED, f"{dataset}_preprocessed.csv")
    
    data = pd.read_csv(path_file, sep=";", header=0)
    # Convert 'Date' and 'Time' columns to a single 'Datetime' column
    data['Datetime'] = pd.to_datetime(data['Date'] + ' ' + data['Time'], format='%d.%m.%Y %H:%M:%S')

    # Set the 'Datetime' column as the index
    data.set_index('Datetime', inplace=True)

    # Drop the now redundant 'Date' and 'Time' columns (we will recreate them later)
    data.drop(columns=['Date', 'Time'], inplace=True)

    # Create a complete time range from 02.09.2024 to 11.09.2024 at 5-second intervals
    full_time_range = pd.date_range(start=start_max, end=end_min, freq='5s')

    # Reindex the DataFrame to include the full time range, introducing NaNs for missing timestamps
    df_full = data.reindex(full_time_range)

    # Recreate the 'Date' and 'Time' columns from the index
    df_full['Date'] = df_full.index.strftime('%d.%m.%Y')
    df_full['Time'] = df_full.index.strftime('%H:%M:%S')

    # Reorder columns so 'Date' and 'Time' come first
    df_full = df_full[['Date', 'Time', 'Temperature', 'Humidity']]
    
    datasets_missing.loc[len(datasets_missing.index)] = [dataset, pd.to_datetime(df_full[df_full['Temperature'].isna()].index[0], format='%d.%m.%Y%H:%M:%S'), pd.to_datetime(df_full[df_full['Temperature'].isna()].index[-1], format='%d.%m.%Y%H:%M:%S'), len(df_full.dropna())]
    
    # Overwrite the original CSV file with the updated DataFrame
    df_full.to_csv(path_file, sep=';', index=False)
    
datasets_missing = datasets_missing.transpose()
    
if 'datasets_overview' not in globals():
    datasets_overview = pd.read_csv(FILE_OVERVIEW, sep=";", header=0, index_col=0)
if 'Missing Start' in datasets_overview.index:
    datasets_overview.drop(index=['Missing Start', 'Missing End'])
datasets_overview = datasets_overview.merge(datasets_missing, on='Sensor', how='left')
datasets_overview.to_csv('overview.csv', sep=";")